# Étapes 2 + 3 : MLflow + Modélisation

## Imports

In [ ]:
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import mlflow.xgboost
import mlflow.lightgbm
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, recall_score, f1_score
import warnings
warnings.filterwarnings('ignore')
import re

## Configuration MLflow

In [ ]:
# création d'un fichier sqlite local pour stocker les résultats des runs
mlflow.set_tracking_uri('sqlite:///mlflow.db')

# création du dossier  qui regroupe tous les runs
mlflow.set_experiment('credit_scoring_v5')

2026/03/19 18:36:36 INFO mlflow.tracking.fluent: Experiment with name 'credit_scoring_v4' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///c:/Users/axell/Documents/projets_code/projet_6/projet6/mlruns/6', creation_time=1773941796882, experiment_id='6', last_update_time=1773941796882, lifecycle_stage='active', name='credit_scoring_v4', tags={}, workspace='default'>

## Chargement dataframes et séparation train test

In [ ]:
# nettoyer les noms de colonnes pour LightGBM (supprimer caractères spéciaux sinon fonctionne pas)

def clean_col_names(df):
    df.columns = [re.sub(r'[^A-Za-z0-9_]', '_', col) for col in df.columns]
    return df

In [ ]:

train = pd.read_csv('C:/Users/axell/Documents/projets_code/projet_6/projet6/train_test/train_final.csv')
test = pd.read_csv('C:/Users/axell/Documents/projets_code/projet_6/projet6/train_test/test_final.csv')

print(f'Shapes finales du train : {train.shape} et test : {test.shape}')

# on sépare les features (X) de la cible (y) et l'ID client (pas utile pour l'entrainement du modèle) + on prépare les features du test
X = train.drop(columns=['TARGET', 'SK_ID_CURR'])
y = train['TARGET']

X = clean_col_names(X)

# On découpe train_final en 80% entraînement et 20% test
# stratify=y pour conserver le même ratio 92%/8% dans les deux parties
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f'shape X_train : {X_train.shape} et shape X_test : {X_test.shape}')

shape X_train : (246008, 581) et shape X_test : (61503, 581)


### Création métrique métier

Pour traduire le déséquilibre du coût business = rater un client qui ne rembourse pas (faux négatif) coûte 10x plus cher que refuser un client qui aurait remboursé (faux positif).

- error_cost : Le modèle ne prédit pas directement "rembourse" ou "rembourse pas" mais une probabilité entre 0 et 1. Par exemple : "ce client a 73% de chances d'être de ne pas rembourser". Pour transformer cette probabilité en décision, il faut un seuil : si proba >= seuil, alors on prédit que le client ne remboursera pas. On compte deux types d'erreurs : FN et FP. Si FN, ça coûte 10, car plus grave, et si FP, ça coûte 1.

- Faux négatif = on a accordé un crédit à quelqu'un qui ne remboursera pas
- Faux positif = on a refusé un crédit à quelqu'un qui aurait remboursé

In [ ]:
def error_cost(y_true, y_proba, threshold=0.5, cost_fn=10, cost_fp=1):
    """
    Calcule le coût total des erreurs pour un seuil donné.
    """
    y_pred = (y_proba >= threshold).astype(int) # pour chaque client, si sa proba de pas rembourser est supérieure au seuil, on le classe comme qq qui remboursera pas (1), sinon qqn qui remboursera (0)
    fn = ((y_true == 1) & (y_pred == 0)).sum() # faux négatifs : clients qui ne rembourseront pas (y_true == 1) mais que le modèle a classé comme remboursant (y_pred == 0)
    fp = ((y_true == 0) & (y_pred == 1)).sum() # faux positifs : clients qui rembourseront (y_true == 0) mais que le modèle a classé comme ne remboursant pas (y_pred == 1)

    return cost_fn * fn + cost_fp * fp # coût total = coût des faux négatifs (10*FN) + coût des faux positifs (1*FP)

### Choix du seuil/threshold

- optimal_threshold : Le seuil par défaut de 0.5 n'est pas forcément le meilleur. Avec un faux négatif (accorder un crédit à qqn qui ne remboursera pas) qui coûte 10x plus qu'un FP il faut être plus sévère = baisser le seuil pour refuser plus de clients dans le doute, même si on rate qq bons clients au passage. Donc on génère des seuils à tester et pour chacun on calcule le coût total. Le meilleur est celui qui retourne le coût le plus bas.
- Seuil trop bas (0.01) = on refuse tout le monde = beaucoup de FP = coût élevé
- Seuil trop haut (0.99) = on accepte tout le monde = beaucoup de FN = coût très élevé

In [ ]:
def optimal_threshold(y_true, y_proba):
    """
    Teste 100 seuils et retourne celui qui coûte le moins cher
    """
    thresholds = np.linspace(0.01, 0.99, 100) #génère des seuils à tester
    costs = [error_cost(y_true, y_proba, t) for t in thresholds] # Pour chaque seuil, on calcule le coût total 10*FN + 1*FP
    best_t = thresholds[np.argmin(costs)] # on trouve l'index du seuil qui minimise le coût
    return best_t, min(costs)  # on retourne ce seuil et le coût associé

## Fonction d'entraînement

1. Ouvre un run MLflow
2. Entraîne le modèle en cross-validation à 5 folds
3. Calcule AUC et coût métier
4. Sauvegarde tout dans MLflow

cross-validation à 5 folds : on découpe le dataset en 5 parties égales et entraîne 5 fois : à chaque fois, 4 parties servent à entraîner et 1 partie sert à évaluer (en rotation).

In [ ]:
def train_and_log(model, model_name, params, X_train, y_train, n_folds=5):
    """
    Entraîne un modèle avec cross-validation et enregistre tout dans MLflow

    Args:
    model: le modèle
    model_name: nom affiché dans MLflow
    params: dictionnaire des hyperparamètres à logger
    n_folds: nb de folds pour la cross-validation
    """

    print(f'\nEntraînement du modèle: {model_name}')

    with mlflow.start_run(run_name=model_name):
        # on lance un run dans mlflow pour ce modèle (toutes les métriques et le modèle lui-même seront enregistrés dans ce run)
        # enregister règlages du modèle dans mlflow
        mlflow.set_tag('model_type', model_name)
        mlflow.log_params(params)
        mlflow.log_param('n_folds', n_folds)

        # stratified k-fold = découpage en folds qui respectent la même proportion de classes que le dataset complet (92% et 8%)
        skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
        oof = np.zeros(len(X_train))  # stocke les prédictions sur chaque fold de validation

        auc_list = []

        for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
            # découper les données en train et validation pour ce fold
            X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
            y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

            # entraîner le modèle sur la partie train du fold
            model.fit(X_tr, y_tr)

            # Prédire les probabilités sur la partie validation
            # predict_proba retourne une proba entre 0 et 1
            val_proba = model.predict_proba(X_val)[:, 1]
            oof[val_idx] = val_proba

            #AUC du fold qu'on ajout dans mlflow
            fold_auc = roc_auc_score(y_val, val_proba)
            auc_list.append(fold_auc)

            # enregistrer l'AUC de ce fold dans MLflow
            mlflow.log_metric('fold_auc', fold_auc, step=fold)
            print(f'Fold {fold+1}/{n_folds} AUC : {fold_auc:.4f}')

            # Seuil optimal pour les métriques de classification
            best_t, _ = optimal_threshold(y_val, val_proba)
            y_pred_opt = (val_proba >= best_t).astype(int)

            # Recall classe minoritaire (TARGET=1 = les défauts)
            # "Sur tous les vrais défauts, combien le modèle en détecte ?"
            recall_minority = recall_score(y_val, y_pred_opt, pos_label=1)

            # F1-score classe minoritaire
            # moyenne harmonique entre précision et recall
            f1_minority = f1_score(y_val, y_pred_opt, pos_label=1)

            mlflow.log_metric('recall_minority', recall_minority, step=fold)
            mlflow.log_metric('f1_minority',f1_minority, step=fold)

        #Métriques finales
        # AUC global sur toutes les parties validation des folds combinées
        auc_global = roc_auc_score(y_train, oof) 
        best_t, cost = optimal_threshold(y_train, oof)

        mlflow.log_metric('auc_global', auc_global)
        mlflow.log_metric('auc_std', np.std(auc_list))
        mlflow.log_metric('error_cost', cost)
        mlflow.log_metric('optimal_threshold', best_t)

        print(f'AUC global =  {auc_global:.4f} (± {np.std(auc_list):.4f})')
        print(f'Coût métier = {cost:.0f}')
        print(f'Meilleur seuil = {best_t:.2f}')

        # Sauvegarder le modèle dans mlflow
        mlflow.sklearn.log_model(model, artifact_path='model', registered_model_name=f'credit_scoring_{model_name.lower().replace(" ", "_")}'
        )

        print(f"{model_name} enregistré dans MLflow")

        # AUC sur le jeu de test pour détecter le surapprentissage (overfit)
        # si auc_global plus grand que auc_test, le modèle a surappris sur le train
        test_proba = model.predict_proba(X_test)[:, 1]
        auc_test = roc_auc_score(y_test, test_proba)
        mlflow.log_metric('auc_test', auc_test)
        print(f'AUC test  = {auc_test:.4f}')

        return auc_global, auc_test, cost, best_t

## Random Forest

In [9]:
rf_params = {
    'n_estimators': 200,  #nb d'arbres
    'max_depth': 10,
    'class_weight' : 'balanced', # compense  déséquilibre des classes
    'random_state': 42,
    'n_jobs' : -1,
}

rf_model = RandomForestClassifier(**rf_params)
rf_auc, rf_auc_test, rf_cost, rf_threshold = train_and_log(rf_model, 'Random Forest', rf_params, X_train, y_train)


Entraînement du modèle: Random Forest
Fold 1/5 AUC : 0.7425
Fold 2/5 AUC : 0.7437
Fold 3/5 AUC : 0.7479
Fold 4/5 AUC : 0.7439
Fold 5/5 AUC : 0.7410


2026/03/19 18:44:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/19 18:44:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


AUC global =  0.7437 (± 0.0023)
Coût métier = 134753
Meilleur seuil = 0.47


Registered model 'credit_scoring_random_forest' already exists. Creating a new version of this model...
Created version '6' of model 'credit_scoring_random_forest'.


Random Forest enregistré dans MLflow
AUC test  = 0.7487


## XGBoost

In [ ]:
# Calcul du ratio pour compenser le déséquilibre
# Si 91% sont 0 et 9% sont 1 -> ratio ≈ 10
ratio = (y == 0).sum() / (y == 1).sum()
print(f'Ratio: {ratio:.1f}')

xgb_params = {
    'n_estimators' : 300,
    'max_depth' : 6,
    'learning_rate' : 0.05,
    'scale_pos_weight': ratio,
    'random_state' : 42,
    'n_jobs' : -1,
    'eval_metric' : 'auc',
    'verbosity': 0,
}

xgb_model = XGBClassifier(**xgb_params)
xgb_auc, xgb_auc_test, xgb_cost, xgb_threshold = train_and_log(xgb_model, 'XGboost', xgb_params, X_train, y_train)

Ratio: 11.4

Entraînement du modèle: XGboost
Fold 1/5 AUC : 0.7772
Fold 2/5 AUC : 0.7808
Fold 3/5 AUC : 0.7789
Fold 4/5 AUC : 0.7807
Fold 5/5 AUC : 0.7789


2026/03/19 18:51:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/19 18:51:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


AUC global =  0.7793 (± 0.0013)
Coût métier = 123164
Meilleur seuil = 0.47


Registered model 'credit_scoring_xgboost' already exists. Creating a new version of this model...
Created version '5' of model 'credit_scoring_xgboost'.


XGboost enregistré dans MLflow
AUC test  = 0.7817


## LightGBM

In [ ]:
lgbm_params = {
    'n_estimators': 300,
    'learning_rate': 0.05,
    'num_leaves' : 31,
    'class_weight': 'balanced',
    'random_state': 42,
    'n_jobs' : -1,
    'verbose' : -1,
}

lgbm_model = lgb.LGBMClassifier(**lgbm_params)
lgbm_auc, lgbm_auc_test, lgbm_cost, lgbm_threshold = train_and_log(lgbm_model, 'LightGBM', lgbm_params, X_train, y_train)


Entraînement du modèle: LightGBM
Fold 1/5 AUC : 0.7802
Fold 2/5 AUC : 0.7833
Fold 3/5 AUC : 0.7819
Fold 4/5 AUC : 0.7848
Fold 5/5 AUC : 0.7832


2026/03/19 18:54:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/19 18:54:43 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


AUC global =  0.7827 (± 0.0015)
Coût métier = 121940
Meilleur seuil = 0.50


Registered model 'credit_scoring_lightgbm' already exists. Creating a new version of this model...
Created version '5' of model 'credit_scoring_lightgbm'.


LightGBM enregistré dans MLflow
AUC test  = 0.7863


## Récupérer les runs depuis MLflow
### Comparaison des résultats

- **AUC** : plus c'est proche de 1 mieux c'est 
- **Coût métier** : plus c'est bas, mieux c'est (objectif principal)
- **Seuil optimal** : si < 0.5, le modèle est sévère (on préfère refuser pour éviter les clients qui ne paieraient pas)

In [33]:
runs = mlflow.search_runs(experiment_names=['credit_scoring_v3'])

cols = ['tags.mlflow.runName', 'metrics.auc_global', 'metrics.auc_test', 'metrics.error_cost', 'metrics.optimal_threshold', 'metrics.recall_minority', 'metrics.f1_minority']
cols = [c for c in cols if c in runs.columns]

print(runs[cols].sort_values('metrics.auc_global', ascending=False).to_string(index=False))

tags.mlflow.runName  metrics.auc_global  metrics.auc_test  metrics.error_cost  metrics.optimal_threshold  metrics.recall_minority  metrics.f1_minority
           LightGBM            0.782484          0.785498            121930.0                   0.514848                 0.653575             0.299337
            XGboost            0.778751          0.781821            123009.0                   0.495051                 0.647281             0.295942
      Random Forest            0.745400          0.750429            134105.0                   0.455455                 0.646777             0.268710


# visualiser dans l'UI MLflow

mlflow ui --backend-store-uri sqlite:///mlflow.db

 ouvrir **http://localhost:5000** 